# **Linear Regression**

**Linear Regression** is a supervised machine learning technique used to model the relationship between one or more input features and a continuous output variable.  
It does this by fitting a linear function to the training data, allowing it to predict numeric values such as house prices, test scores, or measurements.

The function for our linear model is:

$$
f_{w,b}(x^{(i)}) = w \cdot x^{(i)} + b
$$

where:
- $x$ is a feature vector of shape $(n,)$; $n$ features per example  
- $w$ is the weights vector of shape $(n,)$; one weight per feature  
- $b$ is the bias term, a scalar  
- $\cdot$ represents the dot product  
- $x^{(i)}$ is the feature vector for the $i^{\text{th}}$ example

---
Step 1: Let's **compute** the linear regression model, given initial values of w, b and dataset X

In [1]:
def compute_model_output(x,w,b):
  """
  Args:
  x (ndarray): dataset with shape (m,n); m examples and n features
  w (1d-array): weights with shape (n,); n weights
  b (scalar): bias with shape (); 1 value
  
  Returns:
  f_wb (ndarray): output predictions with shape (m,); m predictions
  """

  m, n = x.shape
  f_wb = np.zeros(m)
  
  for i in range(m):
    f_wb[i] = np.dot(w, x[i]) + b
  
  return f_wb

Ideally, our model should minimize the error between its predictions, $f_{w,b}(x)$, and the actual values, $y$.

We can measure this error using the **Mean Squared Error (MSE)** cost function:

$$
J(w, b) = \frac{1}{2m} \sum_{i=0}^{m-1} \left(f_{w,b}(x^{(i)}) - y^{(i)}\right)^2
$$

Here, $J(w, b)$ is the **cost** — a function of our model parameters $w$ and $b$ — which represents the average squared difference between predicted and actual values.

---
Step 2: Compute the cost function $J(w, b)$

In [2]:
def compute_cost(x,y,w,b):
  """
  Args:
  x (ndarray): dataset with shape (m,n); m examples, n features
  y (1d-array): array of output values with shape (m,); m values
  w (ndarray): array of weights with shape (n,); n weights
  b (scalar): bias with shape (); 1 value
  
  Returns:
  J (float): Cost (i.e. model error) from using parameters (w, b) to predict y using x
  """

  m, n = x.shape
  J = 0 # Initialise cost
  f_wb = compute_model_output(x,w,b) # generate predictions for the model
  
  for i in range(m):
    error = (f_wb[i] - y[i]) ** 2
    J += error
  J = (1 / (2 * m)) * J
  
  return J

Having computed the cost function $J(w, b)$ — which measures the model's prediction error — our goal is now to find values of the parameter vector $w$ and scalar $b$ that **minimize** this cost.

This optimization is performed using an iterative method called **Gradient Descent**.

At each iteration, we update the parameters using the following rules:

$$
w \leftarrow w - \alpha \cdot \frac{\partial J(w, b)}{\partial w}
$$

$$
b \leftarrow b - \alpha \cdot \frac{\partial J(w, b)}{\partial b}
$$

**Where:**
- $\alpha$ is the **learning rate**, which controls how large a step we take in the direction of the gradient.
- $\frac{\partial J(w, b)}{\partial w}$ is a **vector of shape $(n,)$**, representing the gradient of the cost with respect to each weight $w_j$.
- $\frac{\partial J(w, b)}{\partial b}$ is a **scalar**, representing the gradient of the cost with respect to the bias term $b$.

Note that w and b converge when gradients $\frac{\partial J(w, b)}{\partial w} \rightarrow 0$ and $\frac{\partial J(w, b)}{\partial b} \rightarrow 0$. 

This is because at this point, the gradient of J(w,b) w.r.t to each of the parameters is 0 - highlighting that we've reached a **global minimum** value of J. We know that it's a global minimum (and not local) because of the nature of the function J.

---

Step 3: Let's **compute** these gradients next.

In [5]:
def compute_gradient(x,y,w,b):
    """
    Args:
    x (ndarray): dataset with shape (m,n); m examples, n features
    y (1d-array): array of output values with shape (m,); m values
    w (ndarray): array of weights with shape (n,); n weights
    b (scalar): bias with shape (); 1 value

    Returns:
    dj_dw (1d-array): array of gradients of cost w.r.t. each weight, with shape (n,); n gradients
    dj_db (scalar): gradient of cost w.r.t. bias with shape (); 1 value
    """
    m,n = x.shape
    dj_dw = np.zeros(n)
    dj_db = 0

    for i in range(m):
        example_error = np.dot(x[i],w) + b - y[i]
        for j in range(n):
            dj_dw[j] = dj_dw[j] + example_error * x[i,j]
        dj_db = dj_db + example_error

    dj_dw = dj_dw / m
    dj_db = dj_db / m

    return dj_dw, dj_db

Now that we have gradients $\frac{\partial J(w, b)}{\partial w}$ and $\frac{\partial J(w, b)}{\partial b}$, we can now run gradient descent to obtain final values of parameters $w$ and $b$!

---

Step 4: **Run** gradient descent.

In [ ]:
def gradient_descent(x, y, w_in, b_in, cost_function, gradient_function, alpha, num_iters): 
    """
    Args:
    x (ndarray): dataset with shape (m,n); m examples, n features
    y (1d-array): array of output values with shape (m, ); m values
    w_in (ndarray): array of initial weights with shape (n, ); n weights
    b (scalar): inital bias with shape (); 1 value
    cost_function: function to compute cost ~ see compute_cost() above
    gradient_function: function to compute gradient ~ see compute_gradient() above
    alpha (float): learning rate for gradient descent
    num_iters (int): number of iterations to run gradient descent

    Returns:
    w (ndarray): final values of parameter w
    b (scalar): final value of parameter b
    """
    w = w_in
    b = b_in
    m,n = x.shape
    cost_history = {}
    
    for i in range(num_iters):
        dj_dw, dj_db = gradient_function(x,y,w,b)
        for j in range(n):
            w[j] -= alpha * dj_dw[j]
        b -= alpha * dj_db

        # print cost at every interval 10-times or every iteration if < 10
        if i% math.ceil(num_iters / 10) == 0: #
            J = compute_cost(x,y,w,b)
            cost_history[i] = J
            print(f"Iteration {i:4d}: Cost {cost_history[-1]:8.2f}   ")

    return w, b, cost_history
    


Step 5 *(Optional)*: To **visualise** how our gradient descent function is performing - we can plot how cost **J** changes as the number of iterations **i** increases.

---

In [1]:
def plot_cost_i_w(hist):
    """
    Args:
    hist (dict): dictionary mapping iteration number to Mean Squared Error during gradient descent

    Returns:
    A plot showing how MSE changes over iterations
    """
    
    iterations = list(hist.keys())
    costs = list(hist.values())

    plt.figure(figsize=(8, 5))
    plt.plot(iterations, costs, marker='o')
    plt.title("Cost vs. Iteration")
    plt.xlabel("Iteration")
    plt.ylabel("Cost")
    plt.grid(True)
    plt.show()